# U8-02 · Perceptrón, retropropagación y clasificación de dígitos

*Machine Learning e Inteligencia Artificial (IFCD0174) · Unidad 8, apartados 8.1 a 8.6*

Ejecuta cada celda con **Mayús + Intro**. Solo se usan NumPy y scikit-learn; no hace falta tarjeta gráfica.

## 1. El perceptrón y su límite

In [ ]:
import numpy as np

def perceptron(X, y, epocas=20, eta=1.0):
    # regla del perceptrón (y en {0, 1}): pesos y fallos por época
    w, b = np.zeros(X.shape[1]), 0.0
    errores = []
    for _ in range(epocas):
        fallos = 0
        for xi, yi in zip(X, y):
            pred = int(w @ xi + b > 0)
            w += eta * (yi - pred) * xi      # solo cambia si falla
            b += eta * (yi - pred)
            fallos += int(pred != yi)
        errores.append(fallos)
    return w, b, errores

X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
for nombre, y in [("Y (AND)", [0, 0, 0, 1]), ("XOR", [0, 1, 1, 0])]:
    w, b, err = perceptron(X, np.array(y))
    print(nombre, "fallos por época:", err[:8], "...", err[-1])

## 2. XOR con una capa oculta (ejemplo 1)

In [ ]:
import numpy as np
from sklearn.neural_network import MLPClassifier

X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y = np.array([0, 1, 1, 0])
red = MLPClassifier(hidden_layer_sizes=(4,), activation="tanh",
                    solver="lbfgs", random_state=0).fit(X, y)
print(red.predict(X), red.predict_proba(X)[:, 1].round(3))

**Prueba.** Cambia `hidden_layer_sizes=(4,)` por `(1,)`. ¿Puede una sola neurona oculta resolver XOR?

## 3. Retropropagación paso a paso (ejemplo 2)

Los mismos números de la figura 8.4. Al final se comprueba un gradiente numéricamente y se da un paso de descenso.

In [ ]:
import numpy as np
sig = lambda z: 1 / (1 + np.exp(-z))

x, y = np.array([1.0, 0.5]), 1.0
W1 = np.array([[0.5, -0.4],          # pesos entrada -> oculta
               [0.3,  0.8]])         # (fila = neurona oculta)
b1 = np.array([0.0, -0.2])
w2, b2 = np.array([0.7, -0.6]), 0.1  # pesos oculta -> salida

# hacia delante
z1 = W1 @ x + b1;  h = sig(z1)
z2 = w2 @ h + b2;  p = sig(z2)
L = -(y * np.log(p) + (1 - y) * np.log(1 - p))   # entropía cruzada

# hacia atrás (regla de la cadena)
d2 = p - y                       # dL/dz2
g_w2, g_b2 = d2 * h, d2
d1 = d2 * w2 * h * (1 - h)       # dL/dz1
g_W1, g_b1 = np.outer(d1, x), d1

print(z1, h.round(4), round(z2, 4), round(p, 4), round(L, 4))
print("d2", round(d2, 4), "g_w2", g_w2.round(4), "d1", d1.round(4))
print("g_W1", g_W1.round(4))
# comprobación numérica de un gradiente
e = 1e-6; W1p = W1.copy(); W1p[0, 0] += e
hp = sig(W1p @ x + b1); pp = sig(w2 @ hp + b2); Lp = -np.log(pp)
print("numérico dL/dW1[0,0]", round((Lp - L) / e, 4))
# un paso de descenso de gradiente con eta = 0.5
eta = 0.5
W1n, b1n = W1 - eta * g_W1, b1 - eta * g_b1
w2n, b2n = w2 - eta * g_w2, b2 - eta * g_b2
pn = sig(w2n @ sig(W1n @ x + b1n) + b2n)
print("tras un paso: p", round(pn, 4), "L", round(-np.log(pn), 4))

**Prueba.** Repite el paso diez veces (un bucle) y observa cómo sube `p` y baja la pérdida.

## 4. Dígitos manuscritos (ejemplo 5)

In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier

X, y = load_digits(return_X_y=True)         # 1797 imágenes de 8x8
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=0)
esc = StandardScaler().fit(X_tr)
A, B = esc.transform(X_tr), esc.transform(X_te)

log = LogisticRegression(max_iter=2000).fit(A, y_tr)
print("logística", round(log.score(B, y_te), 4))
for capas in [(16,), (64,), (128,), (64, 64), (256, 256)]:
    red = MLPClassifier(hidden_layer_sizes=capas, max_iter=500,
                        random_state=0).fit(A, y_tr)
    print(capas, round(red.score(B, y_te), 4), red.n_iter_, "épocas")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay
red = MLPClassifier(hidden_layer_sizes=(64,), max_iter=500,
                    random_state=0).fit(A, y_tr)
pred = red.predict(B)
ConfusionMatrixDisplay.from_predictions(y_te, pred, colorbar=False)
plt.show()

fallos = np.flatnonzero(pred != y_te)
fig, ejes = plt.subplots(1, len(fallos), figsize=(1.3 * len(fallos), 1.8))
for eje, i in zip(np.atleast_1d(ejes), fallos):
    eje.imshow(X_te[i].reshape(8, 8), cmap="gray_r")
    eje.set_title(f"{y_te[i]}→{pred[i]}", fontsize=9); eje.axis("off")
plt.show()

Cada imagen muestra un error: dígito real → dígito predicho.

**Ejercicio.** Entrena la red con 5 semillas distintas (`random_state`) y anota el acierto de cada una. ¿Cuánto varía?

> Rumelhart, D. E., Hinton, G. E. y Williams, R. J. (1986). Learning representations by back-propagating errors. *Nature*, 323, 533–536. https://doi.org/10.1038/323533a0
>
> Alpaydin, E. y Kaynak, C. (1998). Optical Recognition of Handwritten Digits. UCI Machine Learning Repository. https://doi.org/10.24432/C50P49